## News Category Classification Model
### Full Fine-tuning DistilBert model

In [2]:
# generate hugginface token or login to hf account
import huggingface_hub
huggingface_hub.login()

In [3]:
# import the necessary libraries and modules
try:
    import datasets, evaluate, accelerate
    import gradio as gr
except ModuleNotFoundError:
    !pip install datasets evaluate accelerate gradio
    import datasets, evaluate, accelerate
    import gradio as gr

import torch
import transformers
import random
import numpy as np
import pandas as pd

# see version of the libraries
print(f"transformers version: {transformers.__version__}")
print(f"datasets version: {datasets.__version__}")
print(f"evaluate version: {evaluate.__version__}")
print(f"accelerate version: {accelerate.__version__}")
print(f"gradio version: {gr.__version__}")


transformers version: 5.15.1
datasets version: 4.0.0
evaluate version: 0.4.6
accelerate version: 1.14.0
gradio version: 6.26.0


### Prepare Dataset

In [4]:
from datasets import load_dataset

# load the dataset
dataset = load_dataset(path="AiresPucrs/News-Category-Dataset")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 209527
    })
})

In [5]:
print(dataset.column_names)
print(dataset['train'].features)
print(dataset['train'][0])

{'train': ['text', 'labels']}
{'text': Value('string'), 'labels': Value('string')}
{'text': 'Over 4 Million Americans Roll Up Sleeves For Omicron-Targeted COVID Boosters Health experts said it is too early to predict whether demand would match up with the 171 million doses of the new boosters the U.S. ordered for the fall.', 'labels': 'U.S. NEWS'}


In [6]:
# unique labels in the dataset
labels = dataset['train'].unique("labels")
print(f"Unique labels: {labels}")
print(f"Number of unique labels: {len(labels)}")

Unique labels: ['U.S. NEWS', 'COMEDY', 'PARENTING', 'WORLD NEWS', 'CULTURE & ARTS', 'TECH', 'SPORTS', 'ENTERTAINMENT', 'POLITICS', 'WEIRD NEWS', 'ENVIRONMENT', 'EDUCATION', 'CRIME', 'SCIENCE', 'WELLNESS', 'BUSINESS', 'STYLE & BEAUTY', 'FOOD & DRINK', 'MEDIA', 'QUEER VOICES', 'HOME & LIVING', 'WOMEN', 'BLACK VOICES', 'TRAVEL', 'MONEY', 'RELIGION', 'LATINO VOICES', 'IMPACT', 'WEDDINGS', 'COLLEGE', 'PARENTS', 'ARTS & CULTURE', 'STYLE', 'GREEN', 'TASTE', 'HEALTHY LIVING', 'THE WORLDPOST', 'GOOD NEWS', 'WORLDPOST', 'FIFTY', 'ARTS', 'DIVORCE']
Number of unique labels: 42


In [7]:
# turn the dataset into a pandas dataframe and check some samples
news_df = pd.DataFrame(dataset['train'])
news_df.sample(5)

,text,labels
70121,Arianna Huffington On What You Should Never Do...,HEALTHY LIVING
132544,'Smog Buried My Marriage',WORLDPOST
41563,Emma Watson Wore A Subtle 'Beauty And The Beas...,STYLE
76767,Gina Rodriguez Offers Golden Globes Dress To F...,LATINO VOICES
204537,Space Tourism Expected To Be $1 Billion Indust...,TRAVEL


In [8]:
# create mapping of labels to numeric values
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for idx, label in enumerate(labels)}
print(f"Label to ID mapping: {label2id}")
print(f"ID to Label mapping: {id2label}")

Label to ID mapping: {'U.S. NEWS': 0, 'COMEDY': 1, 'PARENTING': 2, 'WORLD NEWS': 3, 'CULTURE & ARTS': 4, 'TECH': 5, 'SPORTS': 6, 'ENTERTAINMENT': 7, 'POLITICS': 8, 'WEIRD NEWS': 9, 'ENVIRONMENT': 10, 'EDUCATION': 11, 'CRIME': 12, 'SCIENCE': 13, 'WELLNESS': 14, 'BUSINESS': 15, 'STYLE & BEAUTY': 16, 'FOOD & DRINK': 17, 'MEDIA': 18, 'QUEER VOICES': 19, 'HOME & LIVING': 20, 'WOMEN': 21, 'BLACK VOICES': 22, 'TRAVEL': 23, 'MONEY': 24, 'RELIGION': 25, 'LATINO VOICES': 26, 'IMPACT': 27, 'WEDDINGS': 28, 'COLLEGE': 29, 'PARENTS': 30, 'ARTS & CULTURE': 31, 'STYLE': 32, 'GREEN': 33, 'TASTE': 34, 'HEALTHY LIVING': 35, 'THE WORLDPOST': 36, 'GOOD NEWS': 37, 'WORLDPOST': 38, 'FIFTY': 39, 'ARTS': 40, 'DIVORCE': 41}
ID to Label mapping: {0: 'U.S. NEWS', 1: 'COMEDY', 2: 'PARENTING', 3: 'WORLD NEWS', 4: 'CULTURE & ARTS', 5: 'TECH', 6: 'SPORTS', 7: 'ENTERTAINMENT', 8: 'POLITICS', 9: 'WEIRD NEWS', 10: 'ENVIRONMENT', 11: 'EDUCATION', 12: 'CRIME', 13: 'SCIENCE', 14: 'WELLNESS', 15: 'BUSINESS', 16: 'STYLE & BE

In [9]:
# Map the labels in the dataset to their corresponding numeric values
def map_labels(example):
    example['labels'] = label2id[example['labels']]
    return example

label_mapped_dataset = dataset.map(map_labels)
# check some sample data after mapping
print(label_mapped_dataset['train'][5:10])

{'text': ['Cleaner Was Dead In Belk Bathroom For 4 Days Before Body Found: Police The 63-year-old woman was seen working at the South Carolina store on Thursday. She was found dead Monday after her family reported her missing, authorities said.', 'Reporter Gets Adorable Surprise From Her Boyfriend While Live On TV "Who\'s that behind you?" an anchor for New York’s PIX11 asked journalist Michelle Ross as she finished up an interview.', 'Puerto Ricans Desperate For Water After Hurricane Fiona’s Rampage More than half a million people remained without water service three days after the storm lashed the U.S. territory.', 'How A New Documentary Captures The Complexity Of Being A Child Of Immigrants In "Mija," director Isabel Castro combined music documentaries with the style of "Euphoria" and "Clueless" to tell a more nuanced immigration story.', "Biden At UN To Call Russian War An Affront To Body's Charter White House officials say the crux of the president's visit to the U.N. this year wi

In [10]:

label_mapped_dataset['train'].shuffle()[:5]


{'text': ["Crystal Renn Shoot Features Unlikely Animal Costars (PHOTOS, VIDEO) Crystal's more conventional work: No stranger to an unconventional photo shoot, Renn has posed pumping gas in lingerie and",
  'The 5 C\'s of White Gold vs. Platinum "Should I set the diamond (or other precious stone) in platinum or white gold?" From my experience in the diamond business for over a decade, here are my thoughts about the pros and cons of white gold versus platinum.',
  "Home Depot's Latest Product Could Save You From Having To Go To Home Depot Again ",
  'Despite Demise, NewsOneNOW Needed More Than Ever With the final live episode of NewsOne Now with Roland Martin in the history books, black America now wakes up to a noticeable',
  'This Week In Pictures: Faith In Practice Around The World, December 28 - January 3 '],
 'labels': [16, 28, 15, 18, 25]}

In [11]:
from datasets import DatasetDict

# split the dataset into train, validation and test sets
train_test_val_dataset = label_mapped_dataset['train'].train_test_split(test_size=0.2, seed=42)
# This results in 10% validation and 10% test relative to the original data
test_valid = train_test_val_dataset["test"].train_test_split(test_size=0.5, seed=42)

# Pack everything into a unified DatasetDict
final_dataset = DatasetDict({
    "train": train_test_val_dataset["train"],
    "validation": test_valid["train"],  # The 'train' part of the second split
    "test": test_valid["test"]          # The 'test' part of the second split
})

print(final_dataset)
final_dataset['test'].shuffle()[:5]


DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 167621
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 20953
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 20953
    })
})


{'text': ["Anti-Islam Group Posts Muslims' Personal Info on Facebook Irving, Texas, is becoming a hotbed of Islamophobia.",
  "Walmart's Safety Vows At Odds With Reality When Walmart’s chief executive, Michael Duke, appeared at a Council on Foreign Relations meeting in New York this month, a",
  "Senate Republicans Call For More Surveillance After Orlando Massacre A new amendment would broaden the FBI's authority to look at email metadata without a warrant.",
  'States Of Emergency Declared Across Australia\'s East Coast As Deadly Fires Loom All residents have been warned “to be on alert."',
  'Sculptor Louise Bourgeois Would Turn 102 If She Were Alive Today Bourgeois, born in Paris in 1911, began studying art in her twenties while enrolled at the renowned French academic institution'],
 'labels': [8, 15, 8, 3, 4]}

### Prepare Tokenizer

In [12]:
from transformers import AutoTokenizer

# load the tokenizer for the model we want to use
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="distilbert/distilbert-base-uncased", use_fast=True)

tokenizer

BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [13]:
# test the tokenizer on a sample text
sample_text = "This is a sample text for tokenization."
tokenized_output = tokenizer(sample_text)
tokenized_output

{'input_ids': [101, 2023, 2003, 1037, 7099, 3793, 2005, 19204, 3989, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [14]:
# check the maximum sequence length of the tokenizer
max_seq_length = tokenizer.model_max_length
vocab_size = tokenizer.vocab_size
print(f"Maximum sequence length of the tokenizer: {max_seq_length}, vocabulary size: {vocab_size}")

Maximum sequence length of the tokenizer: 512, vocabulary size: 30522


In [15]:
# define a tokenization function to apply to the dataset texts
def tokenize_text(example):
    return tokenizer(example['text'], padding=True, truncation=True)

# test the function on a sample text
sample_example= {'text': "The official Google Colab extension for VS Code does not natively support the Secrets (Key icon) user-data feature. Because the extension runs the notebook inside the VS Code Jupyter interface rather than the standard web UI, the google.colab.userdata module will fail to fetch keys stored in your browser-based Colab secrets panel", 'labels': 5}

tokenized_text_sample = tokenize_text(sample_example)
tokenized_text_sample

{'input_ids': [101, 1996, 2880, 8224, 15270, 2497, 5331, 2005, 5443, 3642, 2515, 2025, 3128, 2135, 2490, 1996, 7800, 1006, 3145, 12696, 1007, 5310, 1011, 2951, 3444, 1012, 2138, 1996, 5331, 3216, 1996, 14960, 2503, 1996, 5443, 3642, 18414, 7685, 3334, 8278, 2738, 2084, 1996, 3115, 4773, 21318, 1010, 1996, 8224, 1012, 15270, 2497, 1012, 5310, 2850, 2696, 11336, 2097, 8246, 2000, 18584, 6309, 8250, 1999, 2115, 16602, 1011, 2241, 15270, 2497, 7800, 5997, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [16]:
final_dataset
# Map the tokenization function to the entire dataset
tokenized_dataset = final_dataset.map(function=tokenize_text, batched=True, batch_size=1000)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 167621
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20953
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 20953
    })
})

In [17]:
# see two samples from the tokenized dataset with their all keys
tokenized_dataset['train'].shuffle()[:2]

{'text': ['Parenthesis: The Best of the Parenting Blogosphere I’m all for home births, hospital births, water births, hypnobirths, natural births, and anything else you can dream up that’s',
  'California Democratic Party Chair John Burton On Campaign Finance And The Election (Audio) John Burton is the chair of the California Democratic Party, and he has over three decades of experience working in the California'],
 'labels': [2, 8],
 'input_ids': [[101,
   6687,
   24124,
   1024,
   1996,
   2190,
   1997,
   1996,
   28586,
   9927,
   25444,
   1045,
   1521,
   1049,
   2035,
   2005,
   2188,
   18250,
   1010,
   2902,
   18250,
   1010,
   2300,
   18250,
   1010,
   1044,
   22571,
   25083,
   4313,
   26830,
   1010,
   3019,
   18250,
   1010,
   1998,
   2505,
   2842,
   2017,
   2064,
   3959,
   2039,
   2008,
   1521,
   1055,
   102,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0,
   0

### Evaluate Functions for the Model

In [18]:
import evaluate
import numpy as np
from typing import Tuple

accuracy_metric = evaluate.load("accuracy")

def compute_accuracy(predictions_and_labels: Tuple[np.array, np.array]):
  """
  Computes the accuracy of a model by comparing the predictions and labels.
  """
  predictions, labels = predictions_and_labels

  # Get highest prediction probability of each prediction if predictions are probabilities
  if len(predictions.shape) >= 2:
    predictions = np.argmax(predictions, axis=1)

  return accuracy_metric.compute(predictions=predictions, references=labels)

In [19]:
# Create example list of predictions and labels for testing the evaluate function
example_predictions_all_correct = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
example_predictions_one_wrong = np.array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0])
example_labels = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

# Test the function
print(f"Accuracy when all predictions are correct: {compute_accuracy((example_predictions_all_correct, example_labels))}")
print(f"Accuracy when one prediction is wrong: {compute_accuracy((example_predictions_one_wrong, example_labels))}")

Accuracy when all predictions are correct: {'accuracy': 1.0}
Accuracy when one prediction is wrong: {'accuracy': 0.9}


### Model Training

In [20]:
# define model and load the pretained model for sequence classification
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path= "distilbert/distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [22]:
# count the parameters in the model
def count_params(model):
    """
    Count the parameters of a PyTorch model.
    """
    trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_parameters = sum(p.numel() for p in model.parameters())

    return {"trainable_parameters": trainable_parameters, "total_parameters": total_parameters}

# Count the parameters of the model
count_params(model)

{'trainable_parameters': 66985770, 'total_parameters': 66985770}

In [23]:
# Create model output directory
from pathlib import Path

# Create models directory
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

# Create model save name
model_save_name = "distilbert-base-uncased-News-Category-Classifier"

# Create model save path
model_save_dir = Path(models_dir, model_save_name)

model_save_dir


# define the saved model path (huggingface model hub path)
# model_save_name = "distilbert-base-uncased-News-Category-Classifier"
# model_save_path = f"{huggingface_hub.whoami()['name']}/{model_save_name}"
# model_save_path

PosixPath('models/distilbert-base-uncased-News-Category-Classifier')

In [25]:
# define training arguments for the model training
import torch
from transformers import TrainingArguments

print(f"[INFO] Saving model checkpoints to: {model_save_dir}")

# Detect whether a GPU is actually available. Training on CPU-only vs. a GPU
# needs very different settings (batch size, mixed precision, epoch count),
# so we branch on this rather than guessing.
has_gpu = torch.cuda.is_available()
print(f"[INFO] CUDA available: {has_gpu}")

training_args = TrainingArguments(
    output_dir=model_save_dir,

    # --- learning rate ---
    # 1e-4 is too high for full fine-tuning of a pretrained transformer and risks
    # the loss diverging or the model forgetting its pretrained weights.
    # 2e-5 is the standard starting point for BERT-family fine-tuning.
    learning_rate=2e-5,
    # warmup_ratio=0.1,
    weight_decay=0.01,

    # --- batch size / memory ---
    # Small per-device batch keeps memory usage low on CPU or limited-VRAM GPUs.
    # gradient_accumulation_steps simulates a larger, more stable effective batch
    # (16 * 2 = 32) without needing enough memory to hold a batch of 32 at once.
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,   # eval has no gradients/optimizer state, so it's cheaper
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,     # trades some speed for a large reduction in memory use

    # --- mixed precision ---
    # fp16 only helps (and is only reliably supported) on CUDA GPUs; leave it off
    # on CPU/MPS where it does nothing or isn't supported.
    fp16=has_gpu,

    # --- epochs ---
    # 10 epochs multiplies an already scarce compute budget by 10x and invites
    # overfitting. 3 epochs is a reasonable starting point for fine-tuning on
    # 150k+ examples -- check the eval accuracy curve and extend only if it's
    # still clearly improving.
    num_train_epochs=3,

    # --- evaluation / checkpointing ---
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,              # keep disk usage down (was 3)
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    # --- logging ---
    # Log by steps rather than only at epoch end -- on slow hardware a single
    # epoch can take a long time, and step-level logs confirm training is
    # actually progressing instead of leaving you guessing.
    logging_strategy="steps",
    logging_steps=100,

    # --- misc ---
    dataloader_num_workers=2,        # a couple of background workers speeds up data
                                      # loading without starving CPU-only training of cores
    seed=42,
    report_to="none",                # optional: log to Weights & Biases/similar (off for now)
    # push_to_hub=True,              # optional: automatically upload the model to the Hub
    # hub_token="your_token_here",   # optional: HF token to push (defaults to huggingface-cli login)
    hub_private_repo=False,          # optional: make the uploaded model private
)

effective_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
print(f"[INFO] Effective train batch size: {effective_batch_size}")

[INFO] Saving model checkpoints to: models/distilbert-base-uncased-News-Category-Classifier
[INFO] CUDA available: False
[INFO] Effective train batch size: 32


In [27]:
# define the Trainer for model training and evaluation
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset['validation'],
    processing_class=tokenizer,
    compute_metrics = compute_accuracy
)

trainer

In [28]:
# train the model
train_results = trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 